# BGE-M3 GPU Embedding - HRKP Project (Phase 2)

**Purpose**: CPU 임베딩(~1.3 s/chunk)을 T4 GPU로 가속하여 Dense + Sparse 벡터 동시 생성

**Phase 2 현황** (2026-02-15):
- 전체 청크: 56,063건 (3-Store 정합성 보정 완료)
- 임베딩 필요: 53,414건 (dense_vector 미생성)
- 이미 완료: 2,649건
- 예상 소요: ~9분 (T4 GPU, 100 chunks/s)

**사용법**:
1. Google Drive에 `chunks_for_gpu.jsonl` (53MB) 업로드
2. 런타임 → 런타임 유형 변경 → T4 GPU 선택
3. 전체 셀 실행 (Ctrl+F9)
4. 결과 파일(`chunks_for_gpu_embeddings.jsonl`)을 다운로드하여 import 실행

In [ ]:
# Cell 1: GPU 확인
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: GPU not available! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 2: FlagEmbedding 설치
!pip install -q FlagEmbedding torch

In [ ]:
# Cell 3: Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive 폴더 경로 (내 드라이브 > knowledge_data > documents)
DRIVE_FOLDER = '/content/drive/MyDrive/knowledge_data/documents'

if os.path.exists(DRIVE_FOLDER):
    print(f'Drive folder found: {DRIVE_FOLDER}')
    print(f'Files: {os.listdir(DRIVE_FOLDER)}')
else:
    print(f'Folder not found: {DRIVE_FOLDER}')
    print('Available folders in MyDrive:')
    for f in sorted(os.listdir('/content/drive/MyDrive/'))[:20]:
        print(f'  {f}')

In [ ]:
# Cell 4: BGE-M3 모델 로드 (GPU + FP16)
from FlagEmbedding import BGEM3FlagModel
import time

print('Loading BGE-M3 model on GPU with FP16...')
start = time.time()
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print(f'Model loaded in {time.time()-start:.1f}s')
print(f'Device: {next(model.model.parameters()).device}')

In [ ]:
# Cell 5: 청크 데이터 로드
import json

# Phase 2: 임베딩 미생성 53,414건
INPUT_FILE = 'chunks_for_gpu.jsonl'

input_path = os.path.join(DRIVE_FOLDER, INPUT_FILE)
if not os.path.exists(input_path):
    # Drive 루트에서도 찾기
    input_path = f'/content/drive/MyDrive/{INPUT_FILE}'

print(f'Loading from: {input_path}')

chunks = []
with open(input_path, 'r', encoding='utf-8') as f:
    for line in f:
        chunks.append(json.loads(line.strip()))

texts = [c['text'] for c in chunks]
print(f'Loaded {len(chunks)} chunks')
print(f'Text lengths: min={min(len(t) for t in texts)}, max={max(len(t) for t in texts)}, avg={sum(len(t) for t in texts)/len(texts):.0f}')

In [ ]:
# Cell 6: GPU 임베딩 실행 (Dense + Sparse 동시 생성)
import time
import numpy as np

BATCH_SIZE = 64     # GPU에서 64 배치 (CPU에서는 4)
MAX_LENGTH = 1000   # 기존 데이터 일관성 유지

print(f'Starting GPU embedding: {len(texts)} chunks')
print(f'  batch_size={BATCH_SIZE}, max_length={MAX_LENGTH}, fp16=True')
print(f'  return_dense=True, return_sparse=True')

start = time.time()

result = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False
)

elapsed = time.time() - start
print(f'\nDone: {len(texts)} chunks in {elapsed:.1f}s ({len(texts)/elapsed:.1f} chunks/s)')
print(f'Dense shape: {result["dense_vecs"].shape}')
print(f'Sparse count: {len(result["lexical_weights"])}')

# 샘플 검증
print(f'\nSample dense[0] dim: {len(result["dense_vecs"][0])}')
print(f'Sample sparse[0] keys: {len(result["lexical_weights"][0])}')

In [ ]:
# Cell 7: 결과 저장 (JSONL 형식)
import json
import numpy as np

OUTPUT_FILE = INPUT_FILE.replace('.jsonl', '_embeddings.jsonl')
output_path = os.path.join(DRIVE_FOLDER, OUTPUT_FILE)

print(f'Saving results to: {output_path}')

with open(output_path, 'w', encoding='utf-8') as f:
    for i, chunk in enumerate(chunks):
        # Dense vector: numpy → list
        dense = result['dense_vecs'][i].tolist()

        # Sparse vector: {token_id: weight} dict
        sparse_raw = result['lexical_weights'][i]
        if hasattr(sparse_raw, 'items'):
            sparse = {str(k): float(v) for k, v in sparse_raw.items()}
        else:
            sparse = dict(zip(
                [str(x) for x in sparse_raw.indices.tolist()],
                [float(x) for x in sparse_raw.values.tolist()]
            )) if hasattr(sparse_raw, 'indices') else {}

        record = {
            'chunk_id': chunk['chunk_id'],
            'document_id': chunk['document_id'],
            'dense_vector': dense,
            'sparse_vector': sparse
        }
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

file_size = os.path.getsize(output_path) / 1024 / 1024
print(f'Saved {len(chunks)} embeddings ({file_size:.1f} MB)')
print(f'Output: {output_path}')
print(f'\nDense dim: 1024, Sparse avg keys: {sum(len(result["lexical_weights"][i]) for i in range(len(chunks)))/len(chunks):.0f}')

In [ ]:
# Cell 8: 검증
print('=== Verification ===')

# 첫 번째 결과 확인
with open(output_path, 'r') as f:
    first = json.loads(f.readline())

print(f'chunk_id: {first["chunk_id"]}')
print(f'document_id: {first["document_id"]}')
print(f'dense_vector dim: {len(first["dense_vector"])}')
print(f'dense_vector sample: {first["dense_vector"][:5]}')
print(f'sparse_vector keys: {len(first["sparse_vector"])}')
print(f'sparse_vector sample: {dict(list(first["sparse_vector"].items())[:5])}')

# 전체 통계
dense_norms = [np.linalg.norm(result['dense_vecs'][i]) for i in range(len(chunks))]
print(f'\nDense norm: min={min(dense_norms):.4f}, max={max(dense_norms):.4f}, avg={np.mean(dense_norms):.4f}')
print(f'All norms ~1.0: {all(0.95 < n < 1.05 for n in dense_norms)}')

print(f'\n✅ All {len(chunks)} embeddings generated successfully!')
print(f'📁 Download from: {output_path}')